# JIE42403 — Scientific Visualisation Assignment (Google Colab)

**Title:** *Exploring the Relationship Between Marriage, Live Births, and Population Trends Across Malaysian States (2017–2024)*

**Only 3 visualizations — 1 for each objective.** Each visualization uses **all 3 OpenDOSM datasets**. This notebook retrieves data from **official Parquet URLs**; no CSV upload or HTML export is required.

**Research objectives**
1. To analyze changes in female marriage registrations, live births and population in Malaysia between 2017 and 2024.
2. To examine the association between female marriage rates and crude birth rates among Malaysian states, considering population size.
3. To compare female marriage rates, crude birth rates and population levels across Malaysian states in 2024 using within-indicator standardized values.

**Source pages**
- [Annual Marriages by State](https://open.dosm.gov.my/data-catalogue/marriages_state)
- [Annual Live Births by State](https://open.dosm.gov.my/data-catalogue/births_annual_state)
- [Population Table: States](https://open.dosm.gov.my/data-catalogue/population_state)

*The published API samples using `limit=3` return just three sample rows. We instead read full Parquet files directly from the official hosting URLs.*

## Step 1 — Install libraries and connect to OpenDOSM
Run cells **from top to bottom**. An internet connection is required each time datasets are downloaded.

In [ ]:
%pip -q install "pandas>=2.1,<3" "numpy>=1.26,<3" "plotly>=5.20,<7" "pyarrow>=15,<24"

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

START_YEAR, END_YEAR = 2017, 2024
SOURCES = {
    "marriages": "https://storage.dosm.gov.my/demography/marriages_state.parquet",
    "births": "https://storage.dosm.gov.my/demography/birth_state.parquet",
    "population": "https://storage.dosm.gov.my/population/population_state.parquet",
}
PLOT_COLORS = {"Female marriages": "#2563eb", "Live births": "#0d9488", "Population": "#ea580c"}

try:
    marriages_raw = pd.read_parquet(SOURCES["marriages"])
    births_raw = pd.read_parquet(SOURCES["births"])
    population_raw = pd.read_parquet(SOURCES["population"])
except Exception as exc:
    raise RuntimeError("OpenDOSM download failed. Check internet access or retry the cell.") from exc

print("Original dataset shapes:")
print("Marriage:", marriages_raw.shape, "| Births:", births_raw.shape,
      "| Population:", population_raw.shape)
print("Data sources loaded directly — no manual CSV upload.")

## Step 2 — Clean and merge the three datasets

- Use **female** marriage records to avoid double-counting (male and female records refer to the same registration system but different residence groupings).
- Filter births to the same 2017–2024 window.
- Use only total population: `sex=both`, `age=overall`, `ethnicity=overall`; convert thousands of people to people.
- Merge on `(state, year)`; assert 16 states × 8 years = **128 records**.
- The two rate measures use **different denominators**: female marriage rate per 1,000 unmarried women; crude birth rate per 1,000 total population.

In [ ]:

def prepare_data(marriages_raw: pd.DataFrame, births_raw: pd.DataFrame,
                 population_raw: pd.DataFrame) -> pd.DataFrame:
    """Return unique, validated state-year rows from three official source datasets."""
    def years(frame):
        data = frame.copy()
        data["year"] = pd.to_datetime(data["date"]).dt.year
        data["state"] = data["state"].astype(str).str.strip()
        return data.loc[data["year"].between(START_YEAR, END_YEAR)]

    marriages = years(marriages_raw)
    marriages = (marriages.loc[marriages["sex"].eq("female"),
                              ["state", "year", "abs", "rate"]]
                 .rename(columns={"abs": "female_marriages",
                                  "rate": "female_marriage_rate"}))

    births = (years(births_raw)[["state", "year", "abs", "rate"]]
              .rename(columns={"abs": "live_births", "rate": "crude_birth_rate"}))

    population = years(population_raw)
    population = (population.loc[
        population["sex"].eq("both")
        & population["age"].eq("overall")
        & population["ethnicity"].eq("overall"),
        ["state", "year", "population"]
    ].rename(columns={"population": "total_population"}))
    # OpenDOSM population is in thousands of people; convert to people.
    population["total_population"] = population["total_population"] * 1000

    for title, frame in [("marriages", marriages), ("births", births),
                         ("population", population)]:
        if frame.duplicated(["state", "year"]).any():
            raise ValueError(f"Duplicate state-year keys in {title} source")

    merged = (marriages.merge(births, on=["state", "year"], validate="one_to_one")
              .merge(population, on=["state", "year"], validate="one_to_one")
              .sort_values(["state", "year"]).reset_index(drop=True))
    numeric_cols = ["female_marriages", "female_marriage_rate",
                    "live_births", "crude_birth_rate", "total_population"]
    if merged[numeric_cols].isna().any().any():
        raise ValueError("Missing values found after merging; inspect official source updates")
    if merged["state"].nunique() != 16 or len(merged) != 16 * (END_YEAR - START_YEAR + 1):
        raise ValueError(f"Expected 128 state-year records; found {len(merged)}")
    return merged


master = prepare_data(marriages_raw, births_raw, population_raw)
print(f"Validated: {len(master)} rows, {master.shape[1]} columns, "
      f"{master['state'].nunique()} states, "
      f"{master['year'].min()}–{master['year'].max()}")
display(master.head(8))

## Step 3 — Draw the 3 graphs

**Report recommendation:** Show these three separate charts together on **one A4 portrait page** in the report. In Colab, download each Plotly figure as PNG using its camera icon; no HTML file is generated.

In [ ]:

def chart1_indexed_trends(data: pd.DataFrame, states: list[str]) -> go.Figure:
    """Three-source indexed national or selected-states time trends; 2017=100."""
    subset = data.loc[data["state"].isin(states)]
    yearly = (subset.groupby("year", as_index=False)[
        ["female_marriages", "live_births", "total_population"]].sum())
    lookup = {
        "female_marriages": "Female marriages",
        "live_births": "Live births",
        "total_population": "Population",
    }
    for col, label in lookup.items():
        base = yearly.loc[yearly["year"].eq(START_YEAR), col].iloc[0]
        if base <= 0:
            raise ValueError(f"Cannot index {label} with a zero 2017 baseline")
        yearly[label] = yearly[col] / base * 100
    long = yearly.melt(id_vars="year", value_vars=list(lookup.values()),
                       var_name="Indicator", value_name="Index")
    fig = px.line(long, x="year", y="Index", color="Indicator", markers=True,
                  color_discrete_map=PLOT_COLORS,
                  title="1. Indexed marriage, birth and population trends (2017 = 100)",
                  labels={"year": "Year", "Index": "Index (2017 = 100)"})
    fig.add_hline(y=100, line_dash="dash", line_color="#94a3b8", opacity=0.7)
    fig.update_xaxes(dtick=1, tickformat="d")
    fig.update_layout(hovermode="x unified", legend_title_text="", height=420)
    return fig


def chart2_population_bubbles(data: pd.DataFrame, year: int,
                              states: list[str]) -> tuple[go.Figure, float]:
    """X=marriage rate, Y=birth rate, bubble area=total population."""
    subset = data.loc[data["year"].eq(year) & data["state"].isin(states)].copy()
    subset["Population (millions)"] = subset["total_population"] / 1_000_000
    fig = px.scatter(
        subset, x="female_marriage_rate", y="crude_birth_rate",
        size="total_population", size_max=55, hover_name="state",
        color="state", color_discrete_sequence=px.colors.qualitative.Alphabet,
        hover_data={"Population (millions)": ":.2f", "total_population": False,
                    "female_marriages": ":,.0f", "live_births": ":,.0f", "year": False},
        title=f"2. Marriage–birth rate association; population size ({year})",
        labels={"female_marriage_rate": "Female marriages per 1,000 unmarried women",
                "crude_birth_rate": "Live births per 1,000 people",
                "state": "State"},
    )
    fig.update_traces(marker=dict(opacity=0.75, line=dict(width=1, color="white")))
    fig.update_layout(showlegend=False, height=470)
    corr = subset["female_marriage_rate"].corr(subset["crude_birth_rate"])
    if len(subset) >= 3 and subset["female_marriage_rate"].nunique() > 1:
        slope, intercept = np.polyfit(subset["female_marriage_rate"],
                                      subset["crude_birth_rate"], deg=1)
        xs = np.linspace(subset["female_marriage_rate"].min(),
                         subset["female_marriage_rate"].max(), 100)
        fig.add_trace(go.Scatter(x=xs, y=slope * xs + intercept, mode="lines",
                                 name="Descriptive linear fit", showlegend=False,
                                 line=dict(color="#334155", dash="dash", width=2)))
    return fig, float(corr) if pd.notna(corr) else float("nan")


def chart3_state_heatmap(data: pd.DataFrame, year: int,
                         states: list[str]) -> go.Figure:
    """Z-scores within each 2024 indicator, across currently selected states."""
    subset = (data.loc[data["year"].eq(year) & data["state"].isin(states)]
              .sort_values("state"))
    metrics = {
        "female_marriage_rate": "Marriage rate",
        "crude_birth_rate": "Birth rate",
        "total_population": "Population",
    }
    raw = subset.set_index("state")[list(metrics)].rename(columns=metrics)
    # Standardize across states for each metric, not across mixed units.
    mu = raw.mean(axis=0)
    sd = raw.std(axis=0, ddof=0).replace(0, np.nan)
    zscores = (raw - mu) / sd
    zscores = zscores.fillna(0)
    matrix = zscores.to_numpy(dtype=float)
    details = np.empty(matrix.shape, dtype=object)
    for j, col in enumerate(raw.columns):
        for i in range(len(raw)):
            details[i, j] = (f"{raw.iloc[i, j]:,.0f} people" if col == "Population"
                              else f"{raw.iloc[i, j]:.2f} per 1,000")
    fig = go.Figure(go.Heatmap(
        z=matrix, x=list(raw.columns), y=list(raw.index), customdata=details,
        colorscale="RdBu", reversescale=True, zmid=0, zmin=-2.5, zmax=2.5,
        colorbar=dict(title="Z-score"),
        text=np.round(matrix, 1).astype(str), texttemplate="%{text}",
        hovertemplate="State: %{y}<br>Indicator: %{x}<br>Actual: %{customdata}<br>Z: %{z:.2f}<extra></extra>",
    ))
    fig.update_layout(title=f"3. Standardized state comparison ({year}; within-indicator z-scores)",
                      xaxis_title="Indicator", yaxis_title="State", height=620)
    return fig



### Objective 1 — Demographic trends (3 datasets)

An **indexed multi-line chart** gives each series a 2017 baseline of 100. This makes relative percentage changes comparable despite different absolute magnitudes.

In [ ]:
states_all = sorted(master["state"].unique().tolist())
fig1 = chart1_indexed_trends(master, states_all)
fig1.show()

national = master.groupby("year", as_index=False)[["female_marriages", "live_births", "total_population"]].sum()
start = national[national.year.eq(START_YEAR)].iloc[0]
end = national[national.year.eq(END_YEAR)].iloc[0]
for col, label in [("female_marriages", "Female marriages"), ("live_births", "Live births"),
                   ("total_population", "Population")]:
    change = 100 * (end[col] / start[col] - 1)
    print(f"{label}: {change:+.1f}% change from 2017 to 2024")

### Objective 2 — Relationship analysis (3 datasets)

The bubble scatter uses **female marriage rate** (X), **crude birth rate** (Y), and **population** (bubble size). The dashed line and Pearson correlation describe association, **not causation**. Set `YEAR = 2024` for the final assignment figure.

In [ ]:
YEAR = 2024
fig2, pearson_r = chart2_population_bubbles(master, YEAR, states_all)
fig2.show()
print(f"{YEAR} Pearson r (16 states): {pearson_r:+.3f}")
print("Note: population is represented by bubble AREA; rates have distinct denominators.")

### Objective 3 — State comparison (3 datasets)

The heatmap uses **z-scores**, calculated separately for each of the three indicators across the 16 states for 2024. A positive z-score indicates above-average state value on that indicator; it does **not** indicate growth.

In [ ]:
fig3 = chart3_state_heatmap(master, YEAR, states_all)
fig3.show()

snapshot = master[master.year.eq(YEAR)]
for col, label in [("female_marriage_rate", "marriage rate"),
                   ("crude_birth_rate", "birth rate"),
                   ("total_population", "population")]:
    highest = snapshot.loc[snapshot[col].idxmax()]
    lowest = snapshot.loc[snapshot[col].idxmin()]
    print(f"{label}: highest {highest['state']} ({highest[col]:,.2f}); "
          f"lowest {lowest['state']} ({lowest[col]:,.2f})")

## Step 4 — Quality checks and reporting

**Dataset / visualization checks**
- Exactly **3 figures**, all built using merged records from marriage, birth and population.
- 128 state-year rows for the fixed 2017–2024 research window.
- Do not interpret correlations as causal; a marriage in a given year does not necessarily correspond to a birth in the same year.
- State population totals are estimates (apart from census years) and birth/marriage datasets refer to registrations.
- When writing the report, explain the **units** of marriage rate, birth rate and population, and why normalization/standardization is used.

**For the report:** Use the Plotly camera icon to download each graph as PNG, then place the three PNGs together on **one portrait page**. Include one objective statement and one brief interpretation per graph, plus source citations.

**For GitHub:** `File → Save a copy in GitHub` saves this notebook. For Streamlit deployment, upload the separate `app.py` and `requirements.txt` files to the same repository folder. These files also fetch the full OpenDOSM datasets directly, so there is no CSV upload step.
